# 04 - Backend Inference Sanity Checks

This notebook validates end-to-end backend inference logic with real images:

1. Load image and encode to base64.
2. Run `LivenessService.infer()`.
3. Visualize bounding box and landmarks from detector.
4. Confirm liveness score and label behavior.


In [ ]:
# !pip install -q -r /kaggle/working/Face_Anti_Spoofing_Biometric/requirements-kaggle.txt

In [ ]:
from base64 import b64encode
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('/kaggle/working/Face_Anti_Spoofing_Biometric')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from fas.schemas import LivenessInferRequest
from fas.service import LivenessService

## Configure Model and Sample Image

- Set `LIVENESS_MODEL_PATH` only after notebook 02 exports a scripted checkpoint.
- Use any face image from your prepared crop folder or raw dataset.


In [ ]:
import os

SCRIPTED_CHECKPOINT = Path('/kaggle/working/celeba_spoof_training/best_model_scripted.pt')
if SCRIPTED_CHECKPOINT.exists():
    os.environ['LIVENESS_MODEL_PATH'] = str(SCRIPTED_CHECKPOINT)

sample_image = Path('/kaggle/working/celeba_spoof_prepared/crops_80x80/00000000_example.jpg')
print('Model path:', os.environ.get('LIVENESS_MODEL_PATH'))
print('Sample image exists:', sample_image.exists())

In [ ]:
service = LivenessService()

image_bytes = sample_image.read_bytes()
payload = LivenessInferRequest(image_base64=b64encode(image_bytes).decode('utf-8'))
response = service.infer(payload)

print(response.model_dump())

In [ ]:
image = cv2.imread(str(sample_image))
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

viz = image_rgb.copy()
if response.face_bbox_xyxy:
    x1, y1, x2, y2 = response.face_bbox_xyxy
    cv2.rectangle(viz, (x1, y1), (x2, y2), (0, 255, 0), 2)

if response.face_landmarks:
    for x, y in response.face_landmarks:
        cv2.circle(viz, (int(x), int(y)), 2, (255, 0, 0), -1)

plt.figure(figsize=(6, 6))
plt.imshow(viz)
plt.axis('off')
plt.title(f"label={response.liveness_label}, score={response.liveness_score:.3f}")
plt.show()

In [ ]:
# Optional: batch sanity-check over several images
from random import sample

crop_root = Path('/kaggle/working/celeba_spoof_prepared/crops_80x80')
all_images = sorted(crop_root.rglob('*.jpg'))
subset = sample(all_images, min(10, len(all_images))) if all_images else []

results = []
for image_path in subset:
    payload = LivenessInferRequest(image_base64=b64encode(image_path.read_bytes()).decode('utf-8'))
    pred = service.infer(payload)
    results.append(
        {
            'image': str(image_path),
            'face_detected': pred.face_detected,
            'score': pred.liveness_score,
            'label': pred.liveness_label,
            'latency_ms': pred.latency_ms,
        }
    )

results[:3], len(results)